# Build MAP / Rhythm Ablation Datasets

Derives 3 feature-ablation variants from `model_dataset.csv` by dropping specific
columns only \u2014 no rows are filtered, no new features are added, no values change.
Used to test whether predictive performance depends on MAP-derived features,
rhythm-characterisation features, or both.

| Output                          | Columns dropped                                    |
|----------------------------------|-----------------------------------------------------|
| `no_map_dataset.csv`            | 4 MAP episode-duration features                    |
| `no_rhythm_dataset.csv`         | 5 rhythm characterisation features                 |
| `no_map_no_rhythm_dataset.csv`  | both sets (9 total)                                |

All 3 outputs keep the same 1,284 rows (`caseid` + `episode_number` keys) as
`model_dataset.csv`.

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..").resolve()
DATA_FINAL = BASE_DIR / "data" / "final"

model = pd.read_csv(DATA_FINAL / "model_dataset.csv")
print(f"model_dataset.csv loaded: {model.shape}")

model_dataset.csv loaded: (1284, 130)


In [2]:
# 4 MAP episode-duration features
MAP_COLS = ["map_dur_mean", "map_dur_min", "map_dur_slope", "map_dur_std"]

# 5 rhythm characterisation features (total_arrhythmia_burden_sec is intentionally
# NOT in this list -- it was added back after an earlier over-broad removal; see
# the dataset_comparison_table.png footnote for the full rationale)
RHYTHM_COLS = [
    "dominant_rhythm", "episode_beat_count", "episode_beat_type",
    "episode_dominant_rhythm", "episode_rr_cv",
]

missing_map = [c for c in MAP_COLS if c not in model.columns]
missing_rhythm = [c for c in RHYTHM_COLS if c not in model.columns]
assert not missing_map, f"MAP columns not found in model_dataset.csv: {missing_map}"
assert not missing_rhythm, f"Rhythm columns not found in model_dataset.csv: {missing_rhythm}"

no_map = model.drop(columns=MAP_COLS)
no_rhythm = model.drop(columns=RHYTHM_COLS)
no_map_no_rhythm = model.drop(columns=MAP_COLS + RHYTHM_COLS)

print(f"model_dataset.csv:            {model.shape}")
print(f"no_map_dataset.csv:           {no_map.shape}  (-{len(MAP_COLS)} cols)")
print(f"no_rhythm_dataset.csv:        {no_rhythm.shape}  (-{len(RHYTHM_COLS)} cols)")
print(f"no_map_no_rhythm_dataset.csv: {no_map_no_rhythm.shape}  (-{len(MAP_COLS) + len(RHYTHM_COLS)} cols)")

model_dataset.csv:            (1284, 130)
no_map_dataset.csv:           (1284, 126)  (-4 cols)
no_rhythm_dataset.csv:        (1284, 125)  (-5 cols)
no_map_no_rhythm_dataset.csv: (1284, 121)  (-9 cols)


In [3]:
# Sanity check: same rows/keys, no columns added, only the intended columns removed
model_keys = set(zip(model.caseid, model.episode_number))
for name, variant, expected_dropped in [
    ("no_map", no_map, MAP_COLS),
    ("no_rhythm", no_rhythm, RHYTHM_COLS),
    ("no_map_no_rhythm", no_map_no_rhythm, MAP_COLS + RHYTHM_COLS),
]:
    variant_keys = set(zip(variant.caseid, variant.episode_number))
    dropped = sorted(set(model.columns) - set(variant.columns))
    added = sorted(set(variant.columns) - set(model.columns))
    print(f"{name}: keys match model={variant_keys == model_keys}, "
          f"rows={len(variant)}, dropped={dropped}, added={added}")
    assert variant_keys == model_keys
    assert added == []
    assert dropped == sorted(expected_dropped)
print("\nAll checks passed.")

no_map: keys match model=True, rows=1284, dropped=['map_dur_mean', 'map_dur_min', 'map_dur_slope', 'map_dur_std'], added=[]
no_rhythm: keys match model=True, rows=1284, dropped=['dominant_rhythm', 'episode_beat_count', 'episode_beat_type', 'episode_dominant_rhythm', 'episode_rr_cv'], added=[]
no_map_no_rhythm: keys match model=True, rows=1284, dropped=['dominant_rhythm', 'episode_beat_count', 'episode_beat_type', 'episode_dominant_rhythm', 'episode_rr_cv', 'map_dur_mean', 'map_dur_min', 'map_dur_slope', 'map_dur_std'], added=[]

All checks passed.


In [4]:
no_map.to_csv(DATA_FINAL / "no_map_dataset.csv", index=False)
no_rhythm.to_csv(DATA_FINAL / "no_rhythm_dataset.csv", index=False)
no_map_no_rhythm.to_csv(DATA_FINAL / "no_map_no_rhythm_dataset.csv", index=False)

print("Saved:")
print(f"  {DATA_FINAL / 'no_map_dataset.csv'}")
print(f"  {DATA_FINAL / 'no_rhythm_dataset.csv'}")
print(f"  {DATA_FINAL / 'no_map_no_rhythm_dataset.csv'}")

Saved:
  C:\Users\sukka\Downloads\ioh-prediction\data\final\no_map_dataset.csv
  C:\Users\sukka\Downloads\ioh-prediction\data\final\no_rhythm_dataset.csv
  C:\Users\sukka\Downloads\ioh-prediction\data\final\no_map_no_rhythm_dataset.csv
